# 📊 EDA — Customer Support Tickets (Bitext)

**Goal:** explore the [Bitext customer-support dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) to extract the **real distributions** (topics, intents, text length) that will **calibrate a synthetic ticket generator** for the support triage + RAG demo.

**Why grounded synthesis:** instead of inventing tickets from scratch, we measure a real corpus and reproduce its shape — more realistic data, and a defensible methodology.

## 1. Load the dataset
26,872 real-ish customer-support Q&A pairs, 27 intents across 10 categories.

In [1]:
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = ds["train"].to_pandas()
print(df.shape)
df.head()

/Users/vladislavmarinovich/PycharmProjects/saas-support-rag-triage/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 26872/26872 [00:00<00:00, 108023.80 examples/s]

(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## 2. Dataset structure
Columns, types and a look at what each field holds.

In [3]:
df.info()
df["category"].nunique(), df["intent"].nunique()

<class 'pandas.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   flags        26872 non-null  str  
 1   instruction  26872 non-null  str  
 2   category     26872 non-null  str  
 3   intent       26872 non-null  str  
 4   response     26872 non-null  str  
dtypes: str(5)
memory usage: 19.1 MB


(11, 27)

## 3. Triage labels — category & intent distribution
The mix of topics. The proportions here become the category weights of the simulator.

In [4]:
# Proporciones (normalize=True) — esto alimenta los pesos del simulador
df["category"].value_counts(normalize=True).round(3)

category
ACCOUNT         0.223
ORDER           0.148
REFUND          0.111
INVOICE         0.074
CONTACT         0.074
PAYMENT         0.074
FEEDBACK        0.074
DELIVERY        0.074
SHIPPING        0.073
SUBSCRIPTION    0.037
CANCEL          0.035
Name: proportion, dtype: float64

In [5]:
df["intent"].value_counts(normalize=True).round(3)

intent
check_invoice               0.037
complaint                   0.037
contact_customer_service    0.037
edit_account                0.037
switch_account              0.037
check_payment_methods       0.037
contact_human_agent         0.037
delivery_period             0.037
get_invoice                 0.037
newsletter_subscription     0.037
payment_issue               0.037
registration_problems       0.037
cancel_order                0.037
place_order                 0.037
track_refund                0.037
change_order                0.037
check_refund_policy         0.037
create_account              0.037
get_refund                  0.037
review                      0.037
set_up_shipping_address     0.037
delete_account              0.037
delivery_options            0.037
recover_password            0.037
track_order                 0.037
change_shipping_address     0.036
check_cancellation_fee      0.035
Name: proportion, dtype: float64

## 4. Ticket text length
Character length of the customer message — so synthetic tickets feel real.

In [6]:
df["instruction"].str.len().describe().round(1)

count    26872.0
mean        46.9
std         10.9
min          6.0
25%         40.0
50%         48.0
75%         55.0
max         92.0
Name: instruction, dtype: float64

## Complement — Kaggle IT Support tickets
Source: https://www.kaggle.com/datasets/tobiasbueck/multilingual-customer-support-tickets
Calibrates what Bitext lacks: **priority/urgency** + **realistic body length**.

In [7]:
import kagglehub, os

path = kagglehub.dataset_download("tobiasbueck/multilingual-customer-support-tickets")
print(path)
print(os.listdir(path))

100%|██████████| 16.1M/16.1M [00:00<00:00, 22.6MB/s]

Extracting files...


/Users/vladislavmarinovich/.cache/kagglehub/datasets/tobiasbueck/multilingual-customer-support-tickets/versions/14
['dataset-tickets-german_normalized.csv', 'dataset-tickets-german_normalized_50_5_2.csv', 'aa_dataset-tickets-multi-lang-5-2-50-version.csv', 'dataset-tickets-multi-lang-4-20k.csv', 'dataset-tickets-multi-lang3-4k.csv']


Load the largest multilingual file (20k rows) and inspect its schema.

In [8]:
import pandas as pd

df2 = pd.read_csv(os.path.join(path, "dataset-tickets-multi-lang-4-20k.csv"))
print(df2.shape)
print(df2.columns.tolist())
df2.head()

(20000, 15)
['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']


,subject,body,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Unvorhergesehener Absturz der Datenanalyse-Pla...,Die Datenanalyse-Plattform brach unerwartet ab...,Ich werde Ihnen bei der Lösung des Problems he...,Incident,General Inquiry,low,de,Crash,Technical,Bug,Hardware,Resolution,Outage,Documentation,NaN
1,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,Request,Customer Service,medium,en,Feedback,Sales,IT,Tech Support,NaN,NaN,NaN,NaN
2,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,Request,Customer Service,medium,en,Technical,Product,Guidance,Documentation,Performance,Feature,NaN,NaN
3,Krankenhaus-Dienstleistung-Problem,Ein Medien-Daten-Sperrverhalten trat aufgrund ...,Zurück zur E-Mail-Beschwerde über den Sperrver...,Incident,Customer Service,high,de,Security,Breach,Login,Maintenance,Incident,Resolution,Feedback,NaN
4,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",Request,Customer Service,medium,en,Security,Customer,Compliance,Breach,Documentation,Guidance,NaN,NaN


Filter to English, then measure the two things Bitext lacked: priority mix and full ticket length.

In [9]:
en = df2[df2["language"] == "en"]
print(en.shape)
en["priority"].value_counts(normalize=True).round(3)

(11923, 15)


priority
medium    0.415
high      0.383
low       0.201
Name: proportion, dtype: float64

In [10]:
full_text = en["subject"].fillna("") + " " + en["body"].fillna("")
full_text.str.len().describe().round(1)

count    11923.0
mean       416.8
std        232.1
min          5.0
25%        232.0
50%        387.0
75%        572.0
max       1825.0
dtype: float64

## 6. Extracted simulation parameters

**From Bitext** (structure/vocabulary — NOT frequency; balanced by design):
- Category & intent vocabulary → to be mapped to Polaris features
- Agent `response` style → seed for the KB

**From Kaggle IT Support** (English subset, n = 11,923):
- **Priority:** low / medium / high. Raw 20/42/38 is high-skewed (synthetic) → simulator uses a realistic skew (~55/30/15 — most tickets are routine).
- **Ticket length (subject+body):** mean ≈ 417, median ≈ 387, typical 230–570 chars, max ≈ 1825. Synthetic tickets target this range.
- **Routing:** `queue` values → escalation targets.

**Chosen (design, not measured):**
- Daily volume: ~20–40 tickets/day (plausible + cheap for the live cron).
- % resolvable from KB vs escalate: TBD with Polaris product logic.

**Pending — Polaris product ficha:**
- Final category weights (categories = Polaris features).
- KB article set (from responses + product features).

In [ ]:
### Ticket TYPE taxonomy (from tags)
Flatten tag_1..tag_8 to get the vocabulary of issue *types* (bug, outage, how-to…). This is the "type" axis; the "topic" axis comes from Polaris features.

In [11]:
tag_cols = [c for c in en.columns if c.startswith("tag_")]
tags = pd.Series(en[tag_cols].values.ravel()).dropna()
tags.value_counts().head(30)

Tech Support     4847
IT               4732
Documentation    4232
Feedback         3910
Performance      3887
Bug              3578
Technical        3531
Security         2791
Resolution       2410
Feature          2051
Guidance         1874
Product          1406
Network          1092
Customer         1067
Crash            1050
Outage           1020
Integration      1006
Sales             995
Disruption        821
Billing           816
Incident          748
Breach            615
Payment           569
Maintenance       558
Hardware          510
Software          468
Follow-Up         455
Account           455
Strategy          350
Marketing         349
Name: count, dtype: int64

### Read real tickets — absorb the writing style
Sample a few English tickets to capture tone, structure and specificity — this becomes the style guide for the synthetic generator.

In [13]:
sample = en.sample(10, random_state=7)
for _, r in sample.iterrows():
    print("─" * 70)
    print(f"PRIORITY: {r['priority']}  |  TAGS: {r['tag_1']}, {r['tag_2']}")
    print(f"SUBJECT: {r['subject']}")
    body = str(r['body'])[:400]
    print(f"BODY: {body}")

──────────────────────────────────────────────────────────────────────
PRIORITY: high  |  TAGS: Security, Bug
SUBJECT: Detected Security Breach in Hospital Systems
BODY: A security breach has been detected in the hospital systems, impacting the protection of medical data due to outdated software and firewall issues.
──────────────────────────────────────────────────────────────────────
PRIORITY: high  |  TAGS: Technical, Product
SUBJECT: Requirements for Optimal Performance of SaaS Project Management Tool
BODY: Could you please specify the system requirements needed for the best performance of your SaaS project management application? I aim to make sure that our team's computers are equipped with the necessary specifications.
──────────────────────────────────────────────────────────────────────
PRIORITY: high  |  TAGS: Billing, Payment
SUBJECT: Request for Updating Billing Automation Tools
BODY: I am requesting an update to the billing automation tools within the financial analytics p

### Findings from reading real tickets
- **Domains are scattered** (hospitals, cameras, finance) → this dataset is generic, not one product. We reuse **style, not content** — synthetic tickets are Polaris-only.
- **Texts are too clean / AI-like** (no typos, polished, vague) → our generator adds realism: vary politeness, typos, frustration.
- **Priority labels are noisy** (a "system requirements" question tagged *high*) → we assign priority by **type logic**, not copied from source.
- **Technical tickets follow a pattern:** *problem → what I already tried (cleared cache, restarted, checked logs) → request*. Worth replicating for bug/outage types.
- These real examples double as **few-shot seeds** for the generator prompt.